# 00b. Limpeza de dados

Entre o empilhamento (NB00) e a EDA (NB01). Duas classes de problema:

**Linha que não pode casar.** Quem morreu antes do Censo 2022 não foi enumerado,
então nenhum par Censo × CPF com esse registro é verdadeiro. Sai da base.

**Valor-sentinela que o SQL lê como igualdade.** `''` e `'00000000'` são ausência
escrita como texto, mas `l.cep = r.cep` os trata como acordo real. Isso infla os
blocos e, nas `deterministic_rules` do NB02, contamina a estimativa do prior.
Viram `NULL`, que não casa com `NULL`.

Saída: `registro_limpo` e o parquet correspondente. O `materialize_splink_input`
passa a apontar para ela sozinho, então NB02 e NB03 não mudam.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

from IPython.display import display

PROB_DIR = Path.cwd()
if PROB_DIR.name == 'notebooks':
    PROB_DIR = PROB_DIR.parent
if str(PROB_DIR) not in sys.path:
    sys.path.insert(0, str(PROB_DIR))

import config
from config import (
    COHORT_DEDUP_ARQUIVO,
    COLUNAS_ESTRUTURAIS,
    REGISTRO_LIMPO,
    TABELA_LIMPA,
    cpf_norm_sql,
    dob_valida_sql,
    cep_valido_sql,
    export_parquet,
    get_connection,
    limpeza_columns_sql,
    obito_antes_do_censo_sql,
    print_paths,
    require_tables,
)

ORIGEM = 'registro_unificado'

print_paths()
con = get_connection()
require_tables(con, [ORIGEM], notebook_origem='00')

tipos = {r[0]: r[1] for r in con.execute(f'DESCRIBE {ORIGEM}').fetchall()}
texto_cols = [
    c for c, t in tipos.items()
    if t.upper().startswith('VARCHAR') and c not in COLUNAS_ESTRUTURAIS
]
n_antes = con.execute(f'SELECT COUNT(*) FROM {ORIGEM}').fetchone()[0]
print(f'{ORIGEM}: {n_antes:,} linhas | {len(texto_cols)} colunas de texto a limpar')

## 1. Diagnóstico antes

Quanto de cada coluna é string vazia hoje. São esses valores que o Splink
compara como se fossem iguais entre si.

In [ ]:
vazios = ',\n    '.join(
    f"SUM(CASE WHEN TRIM(CAST({c} AS VARCHAR)) = '' THEN 1 ELSE 0 END) AS {c}"
    for c in texto_cols
)
df_vazios = con.execute(f'''
SELECT origem, COUNT(*) AS n_linhas,
    {vazios}
FROM {ORIGEM} GROUP BY origem ORDER BY origem
''').df().set_index('origem').T
display(df_vazios[df_vazios.sum(axis=1) > 0])

In [ ]:
# Os valores mais frequentes denunciam preenchimento sintético. Se alguma data
# ou CEP aparecer com contagem fora de escala, é sentinela e não dado.
for col in ['data_nascimento', 'cep']:
    print(f'\n=== {col}: 15 valores mais frequentes ===')
    display(con.execute(f'''
    SELECT {col} AS valor, origem, COUNT(*) AS n
    FROM {ORIGEM}
    GROUP BY 1, 2 ORDER BY n DESC LIMIT 15
    ''').df())

In [ ]:
# Distribuição completa de sexo: são poucos valores e cabe inteira. A coluna
# 'mantido' mostra o que sobrevive — o resto ('O' de outro, 'I' de ignorado,
# '9' de não informado) vira NULL. Se alguma categoria fora de M/F tiver volume
# relevante e for real, reveja SEXO_VALIDOS em config.py antes de seguir.
sexo_lista = ', '.join(f"'{v}'" for v in config.SEXO_VALIDOS)
display(con.execute(f'''
SELECT
    CASE WHEN TRIM(CAST(sexo AS VARCHAR)) = '' THEN '(vazio)' ELSE sexo END AS valor,
    origem,
    COUNT(*) AS n,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (PARTITION BY origem), 3) AS pct_origem,
    upper(TRIM(CAST(sexo AS VARCHAR))) IN ({sexo_lista}) AS mantido
FROM {ORIGEM}
GROUP BY 1, 2, 5 ORDER BY origem, n DESC
''').df())

In [ ]:
# Ano de óbito. 'sem óbito' inclui todo o Censo, que nunca tem a coluna.
display(con.execute(f'''
SELECT
    CASE
        WHEN ano_obito IS NULL THEN 'sem ano de óbito'
        WHEN ano_obito <= 0 THEN 'sentinela (<= 0)'
        WHEN ano_obito < {config.ANO_OBITO_CORTE} THEN 'anterior ao corte (sai)'
        WHEN ano_obito > {config.ANO_REFERENCIA_CENSO} THEN 'posterior ao Censo'
        ELSE 'entre o corte e o Censo'
    END AS faixa,
    origem,
    COUNT(*) AS n
FROM {ORIGEM}
GROUP BY 1, 2 ORDER BY n DESC
''').df())

n_obito = con.execute(
    f'SELECT COUNT(*) FROM {ORIGEM} WHERE ano_obito IS NOT NULL'
).fetchone()[0]
if n_obito == 0:
    print(
        'ATENÇÃO: nenhum ano de óbito preenchido, e o NB00 falharia se '
        f'{config.CPF_COL_ANO_OBITO} faltasse no bronze. Então o mais provável é '
        'que este registro_unificado seja anterior à coluna: rode o NB00 de novo. '
        'O filtro de óbito abaixo não vai remover nada.'
    )
else:
    display(con.execute(f'''
    SELECT ano_obito, COUNT(*) AS n FROM {ORIGEM}
    WHERE ano_obito IS NOT NULL GROUP BY 1 ORDER BY ano_obito DESC LIMIT 30
    ''').df())

## 2. Aplicar a limpeza

Um passe só: o `WHERE` remove os óbitos anteriores ao Censo e a projeção troca
os sentinelas por `NULL`. A idade do CPF acompanha a data de nascimento, já que
é derivada dela; a do Censo vem de `PECP0003` e não é tocada.

In [ ]:
cols = limpeza_columns_sql(tipos)
select_sql = ',\n    '.join(
    (alias if expr == alias else f'{expr} AS {alias}') for alias, expr in cols.items()
)
filtro_obito = obito_antes_do_censo_sql()
print('Filtro de óbito:', filtro_obito)

con.execute(f'''
CREATE OR REPLACE TABLE {TABELA_LIMPA} AS
SELECT
    {select_sql}
FROM {ORIGEM}
WHERE NOT {filtro_obito}
''')

n_depois = con.execute(f'SELECT COUNT(*) FROM {TABELA_LIMPA}').fetchone()[0]
print(f'{n_antes:,} → {n_depois:,} linhas ({n_antes - n_depois:,} removidas por óbito)')

colunas_limpa = {r[0] for r in con.execute(f'DESCRIBE {TABELA_LIMPA}').fetchall()}
faltando = set(tipos) - colunas_limpa
if faltando:
    raise RuntimeError(f'Colunas perdidas na limpeza: {sorted(faltando)}')

## 3. Diagnóstico depois

Duas coisas diferentes viram `NULL` e vale separá-las. **Reencodado** é o `''`
que já era ausência e só mudou de grafia. **Descartado** é valor que existia e
foi julgado inválido — data fora da faixa, CEP `00000000`. O segundo número é o
que merece revisão: se estiver alto, a regra pode estar agressiva demais.

In [ ]:
import pandas as pd

linhas = []
for col in texto_cols:
    expr = cols[col]
    if expr == col:
        continue
    linhas.append(con.execute(f'''
    SELECT
        '{col}' AS coluna,
        SUM(CASE WHEN TRIM(CAST({col} AS VARCHAR)) = '' THEN 1 ELSE 0 END) AS reencodado,
        SUM(CASE WHEN TRIM(CAST({col} AS VARCHAR)) <> '' AND ({expr}) IS NULL
                 THEN 1 ELSE 0 END) AS descartado
    FROM {ORIGEM}
    WHERE NOT {filtro_obito}
    ''').df())

resumo = pd.concat(linhas, ignore_index=True)
resumo = resumo[(resumo['reencodado'] > 0) | (resumo['descartado'] > 0)]
display(resumo.sort_values('descartado', ascending=False))

In [ ]:
# Amostra do que foi descartado, para conferir se a regra faz sentido.
display(con.execute(f'''
SELECT data_nascimento, COUNT(*) AS n
FROM {ORIGEM}
WHERE TRIM(CAST(data_nascimento AS VARCHAR)) <> ''
  AND ({dob_valida_sql()}) IS NULL
GROUP BY 1 ORDER BY n DESC LIMIT 20
''').df())

display(con.execute(f'''
SELECT cep, COUNT(*) AS n
FROM {ORIGEM}
WHERE TRIM(CAST(cep AS VARCHAR)) <> ''
  AND ({cep_valido_sql()}) IS NULL
GROUP BY 1 ORDER BY n DESC LIMIT 20
''').df())

In [ ]:
# Preenchimento por origem depois da limpeza: é o que o Splink vai ver.
cobertura = ',\n    '.join(
    f'ROUND(100.0 * COUNT({c}) / COUNT(*), 1) AS {c}' for c in texto_cols
)
display(con.execute(f'''
SELECT origem, COUNT(*) AS n_linhas,
    {cobertura}
FROM {TABELA_LIMPA} GROUP BY origem ORDER BY origem
''').df().set_index('origem').T)

## 4. Impacto na coorte

O filtro de óbito não pode comer ground truth. Se um CPF da coorte tem óbito
anterior a 2021 e mesmo assim aparece no Censo 2022, alguma das duas fontes está
errada — e o par sairia da avaliação do NB03 sem aviso.

In [ ]:
if not COHORT_DEDUP_ARQUIVO.exists():
    print('Coorte não encontrada, checagem pulada:', COHORT_DEDUP_ARQUIVO)
else:
    con.execute(f'''
    CREATE OR REPLACE TEMP TABLE _cohort_cpf AS
    SELECT DISTINCT {cpf_norm_sql('CPF_NORM')} AS cpf_norm
    FROM read_parquet('{COHORT_DEDUP_ARQUIVO}')
    WHERE CPF_NORM IS NOT NULL
    ''')

    display(con.execute(f'''
    WITH na_base AS (
        SELECT u.unique_id FROM {ORIGEM} u
        JOIN _cohort_cpf k ON u.cpf_norm = k.cpf_norm
    )
    SELECT
        COUNT(*) AS cpf_da_coorte_no_subset,
        SUM(CASE WHEN l.unique_id IS NULL THEN 1 ELSE 0 END) AS removidos_por_obito,
        ROUND(100.0 * SUM(CASE WHEN l.unique_id IS NULL THEN 1 ELSE 0 END)
              / NULLIF(COUNT(*), 0), 3) AS pct
    FROM na_base b
    LEFT JOIN {TABELA_LIMPA} l ON l.unique_id = b.unique_id
    ''').df())

    display(con.execute(f'''
    SELECT u.cpf_norm, u.nome_completo, u.data_nascimento, u.ano_obito
    FROM {ORIGEM} u
    JOIN _cohort_cpf k ON u.cpf_norm = k.cpf_norm
    WHERE {filtro_obito}
    LIMIT 20
    ''').df())

## 5. Export

In [ ]:
print('Exportado:', export_parquet(con, TABELA_LIMPA, path=REGISTRO_LIMPO))
con.close()